# 04 · Positional encoding
*Senos y cosenos: cómo sabe el modelo en qué orden vienen los tokens*

Notebook del episodio 4 de **Attention is all you need** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/attention-is-all-you-need/positional-encoding)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** «el perro mordió al hombre» = permutación sin PE → demostración Attn(PX) = P·Attn(X) → fórmula, λ de 2π a ~10000·2π → curva de 4 columnas y matriz pos × dim → identidad del seno de una suma → M_k de rotaciones 2×2 (grilla), error 1e-5 en 512 columnas → PE(p)·PE(p+k) = Σcos(kω_i), solo depende de k → tabla aprendida no extrapola; paper: casi igual → RoPE de adelanto

**Cómo correrlo:** con el entorno de `~/Movies/serie-attention/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "attention"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Attention is all you need. Episodio cuatro: positional encoding. Ya tenemos la atención con varias heads. Pero tiene un agujero: no sabe en qué orden vienen los tokens. Hoy lo demostramos, y vemos cómo lo arregla el paper con senos y cosenos, y por qué eligieron justo esa fórmula.*

## Parte 1 · La atención no ve el orden

Una RNN, recurrent neural network, sabe el orden porque lee de a uno. La atención mira todo a la vez. ¿Qué pasa si cambiamos el orden de las palabras?

### El perro mordió al hombre

Dos frases con las mismas palabras y significado opuesto. Pasan por una tabla de embeddings y una capa de multi-head attention de PyTorch, con pesos al azar. El perro mordió al hombre. El hombre mordió al perro. Comparamos la salida de mordió, que está en la posición dos en las dos frases. Y comparamos la salida de la frase b con la de la frase a reordenada: perro y hombre intercambiados. Sin positional encoding, mordió sale idéntico en las dos frases, y la salida de b es exactamente la de a permutada: True y True. Con positional encoding, las dos dan False. Sin él, para la atención las dos frases son la misma bolsa de palabras.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_permutacion.py
__file__ = 'snippets/ep04_permutacion.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import math
import torch
import torch.nn as nn
torch.manual_seed(0)
d = 16
emb = nn.Embedding(10, d)
attn = nn.MultiheadAttention(d, 2, batch_first=True)
pe = torch.zeros(5, d)
p = torch.arange(5).unsqueeze(1); om = 1 / 10000 ** (torch.arange(0, d, 2) / d)
pe[:, 0::2], pe[:, 1::2] = torch.sin(p * om), torch.cos(p * om)
vocab = {"el": 0, "perro": 1, "mordió": 2, "al": 3, "hombre": 4}
def ids(frase):
    return torch.tensor([[vocab[w] for w in frase.split()]])
def salida(frase, con_pe):
    x = emb(ids(frase)) + (pe if con_pe else 0)
    return attn(x, x, x, need_weights=False)[0][0]

In [3]:
a = "el perro mordió al hombre"
b = "el hombre mordió al perro"
for con_pe in (False, True):
    sa, sb = salida(a, con_pe), salida(b, con_pe)
    igual = torch.allclose(sa[2], sb[2], atol=1e-6)  # «mordió», pos 2
    perm = torch.allclose(sa[[0, 4, 2, 3, 1]], sb, atol=1e-6)
    print(f"PE={con_pe!s:5}  «mordió» igual en las dos: {igual!s:5}"
          f"  b = a permutada: {perm}")

PE=False  «mordió» igual en las dos: True   b = a permutada: True
PE=True   «mordió» igual en las dos: False  b = a permutada: False


### Por qué: permutar entra y sale

$$
X' = P\,X
$$
<small>$P$: matriz de permutación, reordena las filas</small>

$$
(PQ)(PK)^\top = P\, Q K^\top P^\top
$$
<small>la transpuesta de un producto invierte el orden</small>

$$
\mathrm{softmax}(P S P^\top) = P\, \mathrm{softmax}(S)\, P^\top
$$
<small>softmax por filas no ve el orden de las filas ni de las columnas</small>

$$
(P A P^\top)(P V) = P\, (A V)
$$
<small>porque $P^\top P = I$: la salida, permutada igual</small>

La demostración, paso a paso. Reordenar las palabras es multiplicar X por una matriz de permutación P, de ceros y unos, que copia cada fila a otro lugar. En el dibujo, hombre y perro cambian de fila. Q y K salen de multiplicar X por matrices fijas, así que llegan permutadas igual. Q por K transpuesta queda P por Q K transpuesta por P transpuesta: P reordena las filas de la tabla, P transpuesta las columnas. El softmax trabaja fila por fila y no le importa el orden dentro de la fila: en el dibujo, la fila de mordió con perro y hombre intercambiados; cada peso sigue pegado a su palabra. Y como P transpuesta por P es la identidad, I, la salida es P por la salida original. Eso se llama equivarianza: cada token recibe el mismo vector, esté donde esté.

> **Vaswani et al., 2017 — Attention Is All You Need, sección 3.5** · [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
>
> **dim(PE) = d_model = 512** · «Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of the tokens.»

El paper lo plantea así. Como el modelo no tiene recurrencia ni convoluciones, para que pueda usar el orden hay que inyectarle información sobre la posición de cada token, absoluta o relativa. Lo hacen sumando al embedding de cada token un vector de la misma dimensión, d model, quinientos doce: el positional encoding. Se suma, no se concatena: el vector del token no cambia de tamaño.

## Parte 2 · Senos y cosenos

Hay muchas formas de armar ese vector. El paper eligió una fija, sin nada que entrenar.

### La fórmula

$$
PE_{(pos,\,2i)} = \sin\!\big(pos \cdot \omega_i\big),\quad \omega_i = \frac{1}{10000^{2i/d_{\text{model}}}}
$$
<small>columnas pares: seno</small>

$$
PE_{(pos,\,2i+1)} = \cos\!\big(pos \cdot \omega_i\big)
$$
<small>columnas impares: coseno, misma frecuencia</small>

$$
\lambda_i = \frac{2\pi}{\omega_i}:\; 2\pi \;\to\; \approx 10000 \cdot 2\pi
$$
<small>longitudes de onda en progresión geométrica</small>

pos es la posición del token: cero, uno, dos. i numera los pares de columnas: va de cero a doscientos cincuenta y cinco. La columna dos i es el seno de pos por omega sub i, una frecuencia: uno dividido diez mil elevado a dos i sobre d model. Para i igual a cero, omega vale uno: en el dibujo, el seno da una vuelta cada dos pi, seis coma veintiocho posiciones. La columna siguiente, dos i más uno, es el coseno con la misma frecuencia. En el dibujo, la posición uno: seno de uno, cero coma ochenta y cuatro; coseno de uno, cero coma cincuenta y cuatro. Cada par es más lento que el anterior. La longitud de onda, cuántas posiciones tarda en repetirse, crece multiplicándose por el mismo factor: de dos pi hasta casi diez mil por dos pi.

### El positional encoding en PyTorch

Las posiciones, como una columna: cien por uno. Las doscientas cincuenta y seis frecuencias. pos por omega es un producto exterior: cada posición por cada frecuencia. El seno va a las columnas pares, con paso dos en el slicing. El coseno, a las impares. Cien posiciones por quinientas doce. La posición cero es cero, uno, cero, uno: seno y coseno de cero. La uno empieza con cero coma ochenta y cuatro y cero coma cincuenta y cuatro. Las longitudes de onda van de seis coma veintiocho a sesenta mil seiscientos once; el paper redondea a diez mil por dos pi, sesenta y dos mil ochocientos.

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_formula.py
__file__ = 'snippets/ep04_formula.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import math
import torch
torch.set_printoptions(precision=3, sci_mode=False)

In [5]:
def positional_encoding(n_pos, d):
    pos = torch.arange(n_pos).unsqueeze(1)          # [n_pos, 1]
    i2 = torch.arange(0, d, 2)                      # 0, 2, 4, …, d−2
    omega = 1 / 10000 ** (i2 / d)                   # una frecuencia por par
    PE = torch.zeros(n_pos, d)
    PE[:, 0::2] = torch.sin(pos * omega)            # columnas pares
    PE[:, 1::2] = torch.cos(pos * omega)            # columnas impares
    return PE, omega

PE, omega = positional_encoding(100, 512)
print("PE", tuple(PE.shape))
print("pos 0, dims 0–3:", PE[0, :4])
print("pos 1, dims 0–3:", PE[1, :4])
print(f"longitud de onda: {2 * math.pi / omega[0]:.2f}"
      f" … {2 * math.pi / omega[-1]:,.0f}")

PE (100, 512)
pos 0, dims 0–3: tensor([0., 1., 0., 1.])
pos 1, dims 0–3: tensor([0.841, 0.540, 0.822, 0.570])
longitud de onda: 6.28 … 60,611


### Cuatro columnas, cuatro velocidades

Cuatro columnas pares, dibujadas a lo largo de las posiciones. La columna cero oscila rápido: una vuelta cada seis posiciones. La treinta y dos, cada once. La sesenta y cuatro, cada veinte. Y la ciento veintiocho tarda sesenta y tres posiciones: en este tramo no llega a dar una vuelta. Es como un reloj con agujas de distintas velocidades: la rápida distingue posiciones vecinas, las lentas distinguen el principio de la frase del final.

### La matriz posición × dimensión (d = 16)

Ahora toda la matriz, con un d model de dieciséis para que se vea. Cada fila es el vector que se suma a un token según su posición. Las columnas de la izquierda cambian de color de una fila a la otra: son las rápidas. Las de la derecha casi no cambian en diez posiciones: los senos quedan cerca de cero y los cosenos cerca de uno. Ninguna fila se repite: cada posición tiene su huella.

## Parte 3 · PE(pos + k), lineal en PE(pos)

El paper justifica la elección con una propiedad: moverse k posiciones es una cuenta lineal. Vamos a demostrarla.

### La suma de ángulos

$$
\sin(a + b) = \sin a \cos b + \cos a \sin b
$$
<small>con $a = \omega\, pos$, $b = \omega k$: $1$ y $1$</small>

$$
\cos(a + b) = \cos a \cos b - \sin a \sin b
$$
<small>la misma cuenta para el coseno</small>

$$
\begin{pmatrix} \sin \omega(pos{+}k) \\ \cos \omega(pos{+}k) \end{pmatrix} = \begin{pmatrix} \cos \omega k & \sin \omega k \\ -\sin \omega k & \cos \omega k \end{pmatrix} \begin{pmatrix} \sin \omega\, pos \\ \cos \omega\, pos \end{pmatrix}
$$
<small>una rotación que depende solo de $k$</small>

Un solo par de columnas, con frecuencia omega. El seno de una suma: seno de a más b es seno de a por coseno de b, más coseno de a por seno de b, con a igual a omega por pos y b igual a omega por k. En el dibujo, omega uno, posición uno, k uno: cero coma cuatro cinco cinco más cero coma cuatro cinco cinco, cero coma nueve cero nueve, el seno de dos. Lo mismo con el coseno, con un signo menos: menos cero coma cuatro uno seis, el coseno de dos. Con matrices: el par en pos más k es una matriz de dos por dos, M sub k, por el par en pos. M sub k tiene cosenos y senos de omega por k: depende de k, no de pos. Es una rotación.

### Mₖ rota el par (seno, coseno)

La rotación, dibujada, para la columna cero, donde omega vale uno. En verde, el par de la posición uno, cero coma ochenta y cuatro y cero coma cincuenta y cuatro, agrandado tres veces para que se vea; como todo es lineal, la escala no cambia nada. M sub uno gira todo el plano un radián en el sentido de las agujas del reloj: la grilla gira sin deformarse, y el vector verde cae en el par de la posición dos: cero coma nueve uno, menos cero coma cuatro dos, por tres. M sub dos gira dos radianes: la posición uno va a la tres. Moverse k posiciones es girar un ángulo omega por k, en cada par, con una velocidad distinta.

### Verificarlo en las 512 columnas

Armamos M sub k entera: una matriz de quinientos doce por quinientos doce, con un bloque de rotación de dos por dos por cada frecuencia en la diagonal, y ceros en el resto. Los cosenos y senos de omega por k van a las cuatro posiciones de cada bloque. Multiplicamos todas las posiciones por M sub k de una. Y medimos la diferencia con el positional encoding desplazado k lugares. Para k igual a uno, cinco y veinte, el error máximo es del orden de diez a la menos seis: redondeo de punto flotante. Y el bloque de la frecuencia cero: cero coma cincuenta y cuatro, cero coma ochenta y cuatro, la misma matriz del dibujo.

In [6]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_lineal.py
__file__ = 'snippets/ep04_lineal.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
torch.set_printoptions(precision=3, sci_mode=False)
d, n = 512, 100
pos = torch.arange(n).unsqueeze(1)
omega = 1 / 10000 ** (torch.arange(0, d, 2) / d)
PE = torch.zeros(n, d)
PE[:, 0::2], PE[:, 1::2] = torch.sin(pos * omega), torch.cos(pos * omega)

In [7]:
def M(k):                       # una rotación de 2×2 por cada frecuencia
    c, s = torch.cos(k * omega), torch.sin(k * omega)
    R = torch.zeros(d, d)
    R[0::2, 0::2] = torch.diag(c); R[0::2, 1::2] = torch.diag(s)
    R[1::2, 0::2] = torch.diag(-s); R[1::2, 1::2] = torch.diag(c)
    return R

for k in [1, 5, 20]:
    pred = PE[:n - k] @ M(k).T            # M_k · PE(pos), todas las pos
    err = (pred - PE[k:]).abs().max()
    print(f"k={k:2}: máx |M_k PE(pos) − PE(pos+k)| = {err:.1e}")
print("bloque de M_1 para la frecuencia 0:\n", M(1)[:2, :2])

k= 1: máx |M_k PE(pos) − PE(pos+k)| = 7.2e-06
k= 5: máx |M_k PE(pos) − PE(pos+k)| = 6.2e-06
k=20: máx |M_k PE(pos) − PE(pos+k)| = 5.7e-06
bloque de M_1 para la frecuencia 0:
 tensor([[ 0.540,  0.841],
        [-0.841,  0.540]])


### El producto punto solo depende de la distancia

Una consecuencia que le importa a la atención, que compara con producto punto. La norma de cada vector de posición. Y el producto punto entre la posición p y la p más k, para tres p distintas. Todas las normas dan dieciséis: cada par seno, coseno mide uno, y doscientos cincuenta y seis pares dan raíz de doscientos cincuenta y seis. El producto punto da doscientos cuarenta y nueve coma uno para k igual a uno, ciento ochenta y nueve coma seis para cinco, y ciento cincuenta y siete coma seis para veinte, sea cual sea p. Seno por seno más coseno por coseno es el coseno de la diferencia: queda la suma de los cosenos de k por omega sub i.

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_distancia.py
__file__ = 'snippets/ep04_distancia.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
d, n = 512, 100
pos = torch.arange(n).unsqueeze(1)
omega = 1 / 10000 ** (torch.arange(0, d, 2) / d)
PE = torch.zeros(n, d)
PE[:, 0::2], PE[:, 1::2] = torch.sin(pos * omega), torch.cos(pos * omega)

In [9]:
print("norma de cada PE(pos):", PE.norm(dim=1)[:3].numpy().round(2), "…")
for k in [1, 5, 20]:
    puntos = [PE[p] @ PE[p + k] for p in (0, 10, 50)]
    print(f"k={k:2}: PE(p)·PE(p+k) con p=0, 10, 50 →",
          [round(x.item(), 2) for x in puntos])

norma de cada PE(pos): [16. 16. 16.] …
k= 1: PE(p)·PE(p+k) con p=0, 10, 50 → [249.1, 249.1, 249.1]
k= 5: PE(p)·PE(p+k) con p=0, 10, 50 → [189.6, 189.6, 189.6]
k=20: PE(p)·PE(p+k) con p=0, 10, 50 → [157.57, 157.57, 157.57]


### PE(p) · PE(p + k) según la distancia k

Esa suma, para k de cero a sesenta. Empieza en doscientos cincuenta y seis, la norma al cuadrado, y baja rápido en las primeras posiciones: un token se parece más a sus vecinos que a los lejanos. Las tres marcas son los valores que imprimió el snippet. Para la atención, esto significa que con el positional encoding solo, sin proyecciones, el score entre dos posiciones depende de qué tan lejos están, no de dónde.

> **Vaswani et al., 2017 — Attention Is All You Need, sección 3.5** · [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
>
> **PE_pos+k = f_lineal(PE_pos)** · «We hypothesized it would allow the model to easily learn to attend by relative positions, since for any fixed offset k, PE_pos+k can be represented as a linear function of PE_pos.»

Esto es lo que dice el paper. Eligieron esta función porque suponían que le permitiría al modelo aprender fácil a atender por posición relativa, ya que para cualquier desplazamiento fijo k, PE en pos más k es una función lineal de PE en pos. Lo acabamos de verificar. Fijate el verbo: suponían. La propiedad matemática es un hecho; que el modelo la aproveche es una hipótesis.

## Parte 4 · Fijo o aprendido

La alternativa obvia es no usar ninguna fórmula: una tabla con un vector entrenable por posición.

### Una tabla aprendida no extrapola

Un positional encoding aprendido es otra tabla de embeddings: una fila de quinientos doce números por posición, para frases de hasta quinientos doce tokens. Pedimos la posición cien y la seiscientos. Si la fila no existe, PyTorch tira IndexError. La sinusoidal se calcula para cualquier posición. Doscientos sesenta y dos mil parámetros más. La posición cien anda en las dos. La seiscientos, la tabla no la tiene; la fórmula sí. Que el modelo después la entienda bien es otra cuestión: nunca vio esa posición en el entrenamiento.

In [10]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_aprendido.py
__file__ = 'snippets/ep04_aprendido.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn as nn
torch.manual_seed(0)
d = 512
def sinusoidal(pos, d):
    omega = 1 / 10000 ** (torch.arange(0, d, 2) / d)
    pe = torch.zeros(d)
    pe[0::2], pe[1::2] = torch.sin(pos * omega), torch.cos(pos * omega)
    return pe

In [11]:
aprendido = nn.Embedding(512, d)   # una fila entrenable por posición
print("parámetros extra:", aprendido.weight.numel())

for pos in [100, 600]:             # entrenado con frases de ≤ 512
    try:
        v = aprendido(torch.tensor(pos))
        print(f"aprendido,  pos {pos}: forma {tuple(v.shape)}")
    except IndexError:
        print(f"aprendido,  pos {pos}: IndexError, no hay fila")
    print(f"sinusoidal, pos {pos}: forma {tuple(sinusoidal(pos, d).shape)}")

parámetros extra: 262144
aprendido,  pos 100: forma (512,)
sinusoidal, pos 100: forma (512,)
aprendido,  pos 600: IndexError, no hay fila
sinusoidal, pos 600: forma (512,)


> **Vaswani et al., 2017 — Attention Is All You Need, sección 3.5 y tabla 3 (E)** · [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
>
> **«nearly identical results»** · Con positional embeddings aprendidos «the two versions produced nearly identical results». Eligieron la sinusoidal porque «it may allow the model to extrapolate to sequence lengths longer than the ones encountered during training».

El paper probó las dos. Con positional embeddings aprendidos, las dos versiones dieron resultados casi idénticos: es la fila E de su tabla tres, que leemos en el episodio ocho. Eligieron la sinusoidal porque podría permitir extrapolar a frases más largas que las del entrenamiento. Otra vez, podría: no lo midieron ahí.

### Tres formas de darle el orden

Cómo sigue la historia. El paper suma un vector fijo al embedding, una sola vez, abajo de todo. Gemma usa RoPE, rotary position embedding: en vez de sumar nada, rota la query y la key de cada head, en cada capa, con exactamente las rotaciones de dos por dos que vimos hoy. Así el producto punto entre una query y una key depende directamente de la distancia. Lo vemos entero en el episodio nueve.

## Resumiendo

- Sin posición, la atención es equivariante: permutás y se permuta
- $PE_{(pos,2i)} = \sin(pos\,\omega_i)$, $PE_{(pos,2i+1)} = \cos(pos\,\omega_i)$
- $\omega_i = 10000^{-2i/d}$: longitudes de onda de 6,28 a 60.611
- $PE_{pos+k} = M_k\, PE_{pos}$, $M_k$ rotaciones 2×2 (error 1e-5)
- $PE_p \cdot PE_{p+k}$ depende solo de $k$; aprendido ≈ igual
- Próximo: encoder y decoder, la arquitectura completa